# Phase 1.9: Impact Scoring & Feature Aggregation
Generate ImpactScore for each hotspot and export for frontend.

In [1]:
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
import json

df = pd.read_parquet('exports/clustered_violations.parquet')

# Filter only clustered points (ignore noise)
df_clustered = df[df['cluster_id'] >= 0].copy()


## Compute Cluster Area

In [2]:
def compute_cluster_area(cluster_df):
    pts = cluster_df[['x_m', 'y_m']].dropna().values
    if len(pts) < 3:
        return 100.0 # fallback area in sq meters
    try:
        hull = ConvexHull(pts)
        area = hull.volume # Note: volume of 2D hull is area
        return area if area > 0 else 100.0
    except:
        return 100.0

areas = df_clustered.groupby('cluster_id').apply(compute_cluster_area)
areas.name = 'area_m2'


/var/folders/pq/vgpt272s7ss6fqsks3gf3hc00000gn/T/ipykernel_42973/1866105270.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  areas = df_clustered.groupby('cluster_id').apply(compute_cluster_area)


## Cluster Aggregation

In [3]:
# Entropy helper
def hour_entropy(x):
    counts = x.value_counts(normalize=True)
    return -np.sum(counts * np.log2(counts + 1e-9))

def mode_safe(x):
    m = x.mode()
    return m.iloc[0] if not m.empty else None

def get_junction_name(x):
    # filter out 'No Junction'
    valid = x[x != 'No Junction']
    if not valid.empty:
        return valid.mode().iloc[0]
    return 'Midblock'

# Aggregate
cluster_summary = df_clustered.groupby('cluster_id').agg(
    centroid_lat=('latitude', 'mean'),
    centroid_lon=('longitude', 'mean'),
    violation_count=('id', 'count'),
    dominant_violation=('primary_violation', mode_safe),
    dominant_vehicle=('vehicle_type', mode_safe),
    has_junction_pct=('has_junction', 'mean'),
    mean_severity=('severity_score', 'mean'),
    mean_footprint=('vehicle_footprint', 'mean'),
    repeat_rate=('repeat_plate', lambda x: (x > 1).mean()),
    police_station=('police_station', mode_safe),
    junction_name=('junction_name', get_junction_name),
    temporal_entropy=('hour_ist', hour_entropy)
).reset_index()

cluster_summary = cluster_summary.merge(areas, on='cluster_id')

cluster_summary['cluster_type'] = np.where(cluster_summary['has_junction_pct'] > 0.5, 'Junction Blocking', 'Midblock Spillover')

# Density count per sq km
cluster_summary['density'] = cluster_summary['violation_count'] / (cluster_summary['area_m2'] / 1e6)


## Impact Score Calculation

In [4]:
# Normalize components 0-1
def min_max(s):
    if s.max() == s.min(): return s * 0
    return (s - s.min()) / (s.max() - s.min())

cluster_summary['c_density'] = min_max(cluster_summary['density'])
cluster_summary['c_repeat'] = cluster_summary['repeat_rate']
cluster_summary['c_footprint'] = min_max(cluster_summary['mean_footprint'])
cluster_summary['c_severity'] = min_max(cluster_summary['mean_severity'])
cluster_summary['c_junction'] = cluster_summary['has_junction_pct'] # already 0-1
# Penalty for temporal spread. Entropy max is approx log2(24) = ~4.58. 
# Low entropy = concentrated = blind spot. High entropy = spread = chronic hotspot. 
# We want penalty = 1 - normalized_entropy but the prompt said high entropy -> weight higher.
cluster_summary['c_temporal'] = min_max(cluster_summary['temporal_entropy'])

w1, w2, w3, w4, w5, w6 = 0.30, 0.15, 0.15, 0.20, 0.15, 0.05

cluster_summary['impact_score'] = (
    w1 * cluster_summary['c_density'] +
    w2 * cluster_summary['c_repeat'] +
    w3 * cluster_summary['c_footprint'] +
    w4 * cluster_summary['c_severity'] +
    w5 * cluster_summary['c_junction'] +
    w6 * cluster_summary['c_temporal']
)

# Optional component breakdown JSON
def to_breakdown(row):
    return {
        'density': w1 * row['c_density'],
        'repeat': w2 * row['c_repeat'],
        'footprint': w3 * row['c_footprint'],
        'severity': w4 * row['c_severity'],
        'junction': w5 * row['c_junction'],
        'temporal': w6 * row['c_temporal']
    }
cluster_summary['score_breakdown'] = cluster_summary.apply(to_breakdown, axis=1)

print(f"ImpactScore range: {cluster_summary['impact_score'].min():.4f} - {cluster_summary['impact_score'].max():.4f}")
# Sort by impact score descending
cluster_summary = cluster_summary.sort_values('impact_score', ascending=False)
top3 = cluster_summary.head(3)['police_station'].tolist()
print(f"Top-3 hotspots (by station area): {top3}")

# Save JSON
cluster_summary.to_json('exports/hotspots.json', orient='records', default_handler=str)
print("Saved exports/hotspots.json")


ImpactScore range: 0.0142 - 0.4225
Top-3 hotspots (by station area): ['Ashok Nagar', 'Chamarajpet', 'Mico Layout']
Saved exports/hotspots.json
